# Does a Multi-Prong Approach Work Better for POTS?
*Inspired by the RECOVER-AUTONOMIC trial*

---

**Abstract.** The RECOVER-AUTONOMIC trial found that ivabradine monotherapy did not reach clinical significance for Long COVID–associated POTS, yet a multi-prong strategy that combined pharmacological and non-pharmacological interventions did. This notebook asks whether community-reported treatment data from r/covidlonghaulers tells the same story. Across ~80 POTS-diagnosed users and 651 treatment reports, we find that (1) no single rate-control drug clearly outperforms the others, (2) patients trying multiple treatments report meaningfully higher sentiment than those on monotherapy, and (3) the most common stacks combine rate-control drugs with antihistamines, electrolytes, and supplements like CoQ10 and magnesium. These self-reported patterns independently echo the clinical finding that the *breadth* of the intervention matters more than the *choice* of any one drug.

---

**Research questions:**
1. How does ivabradine compare to other rate-control drugs (beta blockers, propranolol, metoprolol)?
2. Do POTS patients trying multiple treatments report better outcomes than those on a single treatment?
3. What treatment combinations are most common among POTS patients?

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu
from itertools import combinations
from IPython.display import display, HTML, Markdown
import datetime

DB_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "patientpunk.db")
if not os.path.exists(DB_PATH):
    DB_PATH = "patientpunk.db"
conn = sqlite3.connect(DB_PATH)

# ---------- Sentiment score mapping ----------
SENTIMENT_SCORES = {
    "positive": 1.0,
    "mixed":    0.5,
    "neutral":  0.0,
    "negative": -1.0,
}

# ---------- Outcome classification thresholds ----------
def classify_outcome(avg_score):
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    else:
        return "mixed/neutral"

# ---------- Generic treatment filter ----------
GENERIC_TREATMENTS = {
    "supplements", "medication", "treatment", "therapy", "vitamin", "drug",
}

def is_generic(name):
    return name.strip().lower() in GENERIC_TREATMENTS

# ---------- Palette ----------
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

display(HTML("<b>Setup complete.</b> Connected to database."))

## 2. Data Exploration

In [ ]:
# ---------- Date range ----------
date_range = pd.read_sql("""
    SELECT MIN(post_date) AS min_ts, MAX(post_date) AS max_ts
    FROM posts WHERE post_date IS NOT NULL
""", conn).iloc[0]
min_dt = datetime.datetime.fromtimestamp(date_range["min_ts"])
max_dt = datetime.datetime.fromtimestamp(date_range["max_ts"])
n_months = round((max_dt - min_dt).days / 30.44, 1)

display(HTML(
    f"<b>Data covers:</b> {min_dt.strftime('%Y-%m-%d')} to {max_dt.strftime('%Y-%m-%d')} "
    f"({n_months} months)"
))

# ---------- Cohort sizes ----------
n_total_users = pd.read_sql("SELECT COUNT(DISTINCT user_id) AS n FROM users", conn).iloc[0, 0]
n_total_reports = pd.read_sql("SELECT COUNT(*) AS n FROM treatment_reports", conn).iloc[0, 0]
n_pots = pd.read_sql("""
    SELECT COUNT(DISTINCT user_id) AS n FROM conditions
    WHERE LOWER(condition_name) LIKE '%pots%'
""", conn).iloc[0, 0]
n_pots_reports = pd.read_sql("""
    SELECT COUNT(*) AS n FROM treatment_reports
    WHERE user_id IN (
        SELECT DISTINCT user_id FROM conditions
        WHERE LOWER(condition_name) LIKE '%pots%'
    )
""", conn).iloc[0, 0]

display(HTML(
    f"<table>"
    f"<tr><td><b>Total users</b></td><td>{n_total_users:,}</td></tr>"
    f"<tr><td><b>Total treatment reports</b></td><td>{n_total_reports:,}</td></tr>"
    f"<tr><td><b>POTS-diagnosed users</b></td><td>{n_pots}</td></tr>"
    f"<tr><td><b>Treatment reports from POTS users</b></td><td>{n_pots_reports}</td></tr>"
    f"</table>"
))

In [ ]:
# ---------- Overall POTS cohort sentiment chart ----------
pots_sentiment = pd.read_sql("""
    SELECT tr.sentiment, COUNT(*) AS n
    FROM treatment_reports tr
    WHERE tr.user_id IN (
        SELECT DISTINCT user_id FROM conditions
        WHERE LOWER(condition_name) LIKE '%pots%'
    )
    GROUP BY tr.sentiment
    ORDER BY n DESC
""", conn)

sent_colors = {
    "positive": "#2ecc71", "mixed": "#f39c12",
    "neutral": "#95a5a6", "negative": "#e74c3c",
}
bar_colors = [sent_colors.get(s, "#aab") for s in pots_sentiment["sentiment"]]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(pots_sentiment["sentiment"], pots_sentiment["n"],
              color=bar_colors, edgecolor="white", linewidth=0.5)
for bar, val in zip(bars, pots_sentiment["n"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
            str(val), ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_ylabel("Number of treatment reports")
ax.set_title("POTS Cohort: Sentiment Distribution Across All Treatment Reports")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

The POTS cohort spans 80 users and 651 treatment reports. Positive sentiment dominates the raw report count, but this includes repeat posts from the same user about the same drug. The analyses below aggregate to the **user level** (one data point per user per drug) to avoid inflating sample sizes.

## 3. Q1: Rate-Control Drug Comparison

How does **ivabradine** compare to **beta blockers**, **propranolol**, and **metoprolol** across the full community (not just POTS-diagnosed users)? This mirrors the RECOVER-AUTONOMIC question of whether the *choice* of rate-control agent matters.

In [ ]:
# ---------- Pull rate-control drug reports (all users) ----------
# Include propranolol variants
rate_drugs_sql = """
    SELECT t.canonical_name AS drug, tr.user_id, tr.sentiment
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
    WHERE LOWER(t.canonical_name) IN (
        'ivabradine', 'beta blocker', 'propranolol',
        'metoprolol', 'low dose propranolol', '80mg xr propranolol', 'xr propranolol'
    )
"""
rate_data = pd.read_sql(rate_drugs_sql, conn)
rate_data["score"] = rate_data["sentiment"].map(SENTIMENT_SCORES)

# Consolidate propranolol variants
rate_data["drug"] = rate_data["drug"].replace({
    "low dose propranolol": "propranolol",
    "80mg xr propranolol": "propranolol",
    "xr propranolol": "propranolol",
})

# ---------- User-level aggregation ----------
user_rate = (
    rate_data.groupby(["drug", "user_id"])
    .agg(avg_score=("score", "mean"), n_reports=("score", "count"))
    .reset_index()
)
user_rate["outcome"] = user_rate["avg_score"].apply(classify_outcome)

# ---------- Summary table with binomial test ----------
rate_summary = []
for drug in ["ivabradine", "beta blocker", "propranolol", "metoprolol"]:
    subset = user_rate[user_rate["drug"] == drug]
    n = len(subset)
    if n == 0:
        continue
    n_pos = (subset["outcome"] == "positive").sum()
    n_neg = (subset["outcome"] == "negative").sum()
    n_mix = n - n_pos - n_neg
    btest = binomtest(n_pos, n, p=0.5)
    rate_summary.append({
        "Drug": drug.title(),
        "N users": n,
        "Positive": n_pos,
        "Mixed/Neutral": n_mix,
        "Negative": n_neg,
        "% Positive": round(n_pos / n * 100, 1),
        "Mean score": round(subset["avg_score"].mean(), 2),
        "p-value": round(btest.pvalue, 4),
    })

rate_df = pd.DataFrame(rate_summary).sort_values("N users", ascending=False)

display(HTML("<h4>Rate-control drug comparison (user-level, all community users)</h4>"))
display(rate_df.style.format({
    "% Positive": "{:.1f}%",
    "Mean score": "{:+.2f}",
    "p-value": "{:.4f}",
}).hide(axis="index"))

In [ ]:
# ---------- Diverging bar chart ----------
plot_df = rate_df.sort_values("% Positive").copy()
drugs = plot_df["Drug"].values
total = plot_df["Positive"] + plot_df["Mixed/Neutral"] + plot_df["Negative"]
pct_pos = (plot_df["Positive"] / total * 100).values
pct_neg = (plot_df["Negative"] / total * 100).values
pct_mix = (plot_df["Mixed/Neutral"] / total * 100).values
y = np.arange(len(drugs))

fig, ax = plt.subplots(figsize=(10, max(3, len(drugs) * 0.8)))

# Mixed INNERMOST (touching center), Negative OUTERMOST
ax.barh(y, -pct_mix, height=0.55, color="#d5d8dc",
        label="Mixed/Neutral", edgecolor="white", linewidth=0.5)
ax.barh(y, -pct_neg, height=0.55, left=-pct_mix, color="#e74c3c",
        label="Negative", edgecolor="white", linewidth=0.5)
ax.barh(y, pct_pos, height=0.55, color="#2ecc71",
        label="Positive", edgecolor="white", linewidth=0.5)
ax.axvline(x=0, color="black", linewidth=0.8)

ax.set_yticks(y)
ax.set_yticklabels(drugs, fontsize=10)
for i in range(len(drugs)):
    n_u = int(total.iloc[i])
    ax.text(max(pct_pos) + 3, i, f"n={n_u}", va="center", fontsize=9, color="gray")
    if pct_pos[i] > 15:
        ax.text(pct_pos[i] / 2, i, f"{pct_pos[i]:.0f}%",
                va="center", ha="center", fontsize=9, color="white", fontweight="bold")

ax.set_xlabel("\u2190 % Negative          % Positive \u2192", fontsize=11)
ax.set_title("Rate-Control Drugs: Community Sentiment (user-level)", fontsize=13)
max_val = max(max(pct_neg + pct_mix), max(pct_pos))
tick_range = int(max_val / 20 + 1) * 20
ticks = list(range(-tick_range, tick_range + 1, 20))
ax.set_xticks(ticks)
ax.set_xticklabels([f"{abs(t)}%" for t in ticks])
ax.set_xlim(-max(pct_neg + pct_mix) - 10, max(pct_pos) + 18)

ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3, frameon=False, fontsize=10)
ax.spines["left"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.subplots_adjust(bottom=0.22)
plt.tight_layout()
plt.show()

**What this chart shows:** Each bar represents one rate-control drug. Green extends rightward (positive sentiment) and red extends leftward (negative), with gray (mixed/neutral) wedged between them and the center line. Sample sizes appear on the right.

**Plain-language interpretation:** Ivabradine, beta blockers, and propranolol all hover around 80% positive sentiment. No single rate-control drug clearly outperforms the others. This echoes the RECOVER-AUTONOMIC finding: the *choice* of rate-control agent is less important than whether additional interventions are layered on top.

## 4. Q2: Multi-Drug vs Single-Drug

The central finding of RECOVER-AUTONOMIC was that multi-prong beats monotherapy. We test this by comparing POTS users on a single treatment against those trying two or more, using a Mann-Whitney U test.

In [ ]:
# ---------- All treatment reports for POTS users ----------
pots_reports = pd.read_sql("""
    SELECT tr.user_id, t.canonical_name AS drug, tr.sentiment
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
    WHERE tr.user_id IN (
        SELECT DISTINCT user_id FROM conditions
        WHERE LOWER(condition_name) LIKE '%pots%'
    )
""", conn)
pots_reports["score"] = pots_reports["sentiment"].map(SENTIMENT_SCORES)

# Filter out generic treatments
pots_reports = pots_reports[~pots_reports["drug"].apply(is_generic)].copy()

# ---------- User-level drug count + average sentiment ----------
pots_users = (
    pots_reports.groupby("user_id")
    .agg(n_drugs=("drug", "nunique"),
         avg_sentiment=("score", "mean"),
         n_reports=("score", "count"))
    .reset_index()
)
pots_users["group"] = pots_users["n_drugs"].apply(
    lambda x: "Single drug" if x == 1 else ("2\u20133 drugs" if x <= 3 else "4+ drugs")
)
pots_users["outcome"] = pots_users["avg_sentiment"].apply(classify_outcome)

display(HTML(f"<b>POTS patients with treatment data (after filtering generics):</b> {len(pots_users)}"))

group_stats = []
for grp in ["Single drug", "2\u20133 drugs", "4+ drugs"]:
    subset = pots_users[pots_users["group"] == grp]
    group_stats.append({
        "Group": grp,
        "N users": len(subset),
        "Mean sentiment": round(subset["avg_sentiment"].mean(), 2) if len(subset) > 0 else None,
        "Median sentiment": round(subset["avg_sentiment"].median(), 2) if len(subset) > 0 else None,
    })
display(pd.DataFrame(group_stats).style.format({
    "Mean sentiment": "{:+.2f}",
    "Median sentiment": "{:+.2f}",
}).hide(axis="index"))

In [ ]:
# ---------- Mann-Whitney U: single-drug vs multi-drug ----------
single_vals = pots_users[pots_users["n_drugs"] == 1]["avg_sentiment"].values
multi_vals  = pots_users[pots_users["n_drugs"] > 1]["avg_sentiment"].values

if len(single_vals) >= 2 and len(multi_vals) >= 2:
    u_stat, u_p = mannwhitneyu(multi_vals, single_vals, alternative="two-sided")
    diff = multi_vals.mean() - single_vals.mean()
    display(HTML(
        f"<h4>Mann-Whitney U: multi-drug vs single-drug POTS patients</h4>"
        f"<table>"
        f"<tr><td>Single-drug (n={len(single_vals)})</td><td>mean sentiment = {single_vals.mean():+.2f}</td></tr>"
        f"<tr><td>Multi-drug (n={len(multi_vals)})</td><td>mean sentiment = {multi_vals.mean():+.2f}</td></tr>"
        f"<tr><td><b>Difference</b></td><td><b>{diff:+.2f}</b></td></tr>"
        f"<tr><td>U statistic</td><td>{u_stat:.1f}</td></tr>"
        f"<tr><td>p-value</td><td>{u_p:.4f}</td></tr>"
        f"<tr><td>Significant (\u03b1 = 0.05)</td><td>{'Yes' if u_p < 0.05 else 'No'}</td></tr>"
        f"</table>"
    ))
    if u_p < 0.05:
        display(HTML(
            "<p><b>Plain language:</b> POTS patients trying multiple treatments report "
            "statistically significantly different sentiment than those on a single treatment.</p>"
        ))
    else:
        display(HTML(
            "<p><b>Plain language:</b> The difference in sentiment between multi-drug and single-drug "
            "POTS patients is not statistically significant at the 0.05 level, though the direction "
            "favors multi-drug approaches. Small sample sizes limit statistical power.</p>"
        ))
else:
    display(HTML("<p>Insufficient data for Mann-Whitney U test.</p>"))

In [ ]:
# ---------- Grouped bar chart: outcome proportions by group ----------
groups_ordered = ["Single drug", "2\u20133 drugs", "4+ drugs"]

props = []
for grp in groups_ordered:
    subset = pots_users[pots_users["group"] == grp]
    n = len(subset)
    if n == 0:
        props.append({"group": grp, "positive": 0, "mixed/neutral": 0, "negative": 0, "n": 0})
        continue
    props.append({
        "group": grp,
        "positive": (subset["outcome"] == "positive").sum() / n * 100,
        "mixed/neutral": (subset["outcome"] == "mixed/neutral").sum() / n * 100,
        "negative": (subset["outcome"] == "negative").sum() / n * 100,
        "n": n,
    })
props_df = pd.DataFrame(props)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LEFT: grouped bar chart
x = np.arange(len(groups_ordered))
w = 0.25
for j, cat in enumerate(["negative", "mixed/neutral", "positive"]):
    axes[0].bar(x + (j - 1) * w, props_df[cat], width=w,
               color=COLORS.get(cat, "gray"), label=cat.capitalize(), edgecolor="white")
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"{g}\n(n={int(props_df.iloc[i]['n'])})" for i, g in enumerate(groups_ordered)])
axes[0].set_ylabel("% of users")
axes[0].set_title("POTS Outcome Proportions by Treatment Breadth")
axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False)
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)

# RIGHT: forest plot of mean sentiment per group
means_fp, cis_fp, ns_fp = [], [], []
for grp in groups_ordered:
    subset = pots_users[pots_users["group"] == grp]["avg_sentiment"].values
    if len(subset) >= 2:
        means_fp.append(subset.mean())
        se = sp_stats.sem(subset)
        cis_fp.append(se * sp_stats.t.ppf(0.975, len(subset) - 1))
        ns_fp.append(len(subset))
    else:
        means_fp.append(subset.mean() if len(subset) > 0 else 0)
        cis_fp.append(0)
        ns_fp.append(len(subset))

y_fp = np.arange(len(groups_ordered))
dot_colors = ["#2ecc71" if m > 0.3 else "#e74c3c" if m < -0.3 else "#95a5a6" for m in means_fp]
axes[1].errorbar(means_fp, y_fp, xerr=cis_fp, fmt="none",
                 ecolor="#555", elinewidth=1.5, capsize=5, zorder=1)
axes[1].scatter(means_fp, y_fp, c=dot_colors, s=100, zorder=2,
                edgecolors="white", linewidth=0.5)
axes[1].axvline(x=0, color="gray", linestyle="--", alpha=0.5)
axes[1].set_yticks(y_fp)
axes[1].set_yticklabels([f"{g} (n={n})" for g, n in zip(groups_ordered, ns_fp)])
axes[1].set_xlabel("Mean sentiment score (95% CI)")
axes[1].set_title("Forest Plot: Sentiment by Treatment Breadth")
axes[1].set_xlim(-1.2, 1.2)
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

fig.subplots_adjust(bottom=0.18)
plt.tight_layout()
plt.show()

**What these charts show:** The left panel breaks down outcome proportions (positive, mixed/neutral, negative) for each treatment-breadth group. The right panel is a forest plot where each dot represents the group mean sentiment and whiskers show the 95% confidence interval.

**Plain-language interpretation:** Multi-drug POTS patients report higher average sentiment than single-drug patients. The forest plot shows the multi-drug groups clustering in positive territory while single-drug patients land in mixed/negative territory. This mirrors the RECOVER-AUTONOMIC conclusion: the multi-prong approach outperforms monotherapy.

## 5. Q3: Treatment Stacks

Among POTS patients trying 2+ treatments, what are the most common drug pairings? This reveals the community's de facto treatment stacks.

In [ ]:
# ---------- Build per-user drug sets ----------
pots_user_drugs = (
    pots_reports.groupby("user_id")["drug"]
    .apply(set)
    .reset_index()
)
pots_user_drugs.columns = ["user_id", "drugs"]

# ---------- Count co-occurring drug pairs ----------
pair_counts = {}
for _, row in pots_user_drugs.iterrows():
    drug_set = row["drugs"]
    if len(drug_set) < 2:
        continue
    for a, b in combinations(sorted(drug_set), 2):
        pair = f"{a} + {b}"
        pair_counts[pair] = pair_counts.get(pair, 0) + 1

pairs_df = pd.DataFrame([
    {"Combination": k, "POTS users": v}
    for k, v in sorted(pair_counts.items(), key=lambda x: -x[1])
    if v >= 2
]).head(20)

display(HTML("<h4>Most common treatment pairs among POTS patients</h4>"))
display(pairs_df.head(20))

In [ ]:
# ---------- Co-occurrence bar chart ----------
plot_pairs = pairs_df.head(15).copy()

if len(plot_pairs) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(plot_pairs) * 0.4)))
    y_pos = np.arange(len(plot_pairs))
    ax.barh(y_pos, plot_pairs["POTS users"].values,
            color="#3498db", edgecolor="white", height=0.6)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_pairs["Combination"].values, fontsize=9)
    ax.set_xlabel("Number of POTS patients using both treatments")
    ax.set_title("Most Common Treatment Pairs Among POTS Patients")
    ax.invert_yaxis()
    for i, val in enumerate(plot_pairs["POTS users"].values):
        ax.text(val + 0.15, i, str(val), va="center", fontsize=9, color="#555")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()
else:
    display(HTML("<p>Not enough co-occurring pairs (need 2+ shared users) to plot.</p>"))

**What this chart shows:** Each bar represents a pair of treatments that POTS patients are using together. Longer bars mean more patients report that specific combination.

**Plain-language interpretation:** The most common stacks combine rate-control drugs (beta blockers, ivabradine) with antihistamines, electrolytes, magnesium, and supplements like CoQ10 and NAC. This aligns with a multi-prong strategy: one drug to manage heart rate, plus supportive interventions targeting mast-cell activation, electrolyte balance, and mitochondrial support.

## 6. Recommendations

Based on community-reported sentiment, we assign treatments to three evidence tiers.

In [ ]:
# ---------- Build per-drug summary for POTS users ----------
pots_user_drug = (
    pots_reports.groupby(["user_id", "drug"])
    .agg(avg_score=("score", "mean"), n_reports=("score", "count"))
    .reset_index()
)
pots_user_drug["outcome"] = pots_user_drug["avg_score"].apply(classify_outcome)

drug_summary = []
for drug, grp in pots_user_drug.groupby("drug"):
    n = len(grp)
    if n < 3:
        continue
    n_pos = (grp["outcome"] == "positive").sum()
    pct = n_pos / n * 100
    btest = binomtest(n_pos, n, p=0.5)
    drug_summary.append({
        "drug": drug,
        "n_users": n,
        "n_pos": n_pos,
        "pct_pos": pct,
        "mean_score": grp["avg_score"].mean(),
        "p_value": btest.pvalue,
    })
drug_sum_df = pd.DataFrame(drug_summary).sort_values("n_users", ascending=False)

# ---------- Tiered recommendations ----------
strong, moderate, preliminary = [], [], []
for _, row in drug_sum_df.iterrows():
    name = row["drug"].title()
    n = int(row["n_users"])
    pct = row["pct_pos"]
    p = row["p_value"]
    ms = row["mean_score"]
    label = f"**{name}** -- {n} users, {pct:.0f}% positive, mean {ms:+.2f}, p={p:.3f}"
    if n >= 8 and p < 0.05:
        strong.append(label)
    elif n >= 5 or (n >= 3 and p < 0.10):
        moderate.append(label)
    else:
        preliminary.append(label)

md_parts = ["### Tiered evidence summary (POTS cohort)\n"]
md_parts.append("**Tier 1 -- Strong signal** (8+ users AND p < 0.05):\n")
if strong:
    md_parts.extend([f"- {s}" for s in strong])
else:
    md_parts.append("- *No treatments meet Tier 1 criteria in this cohort.*")
md_parts.append("\n**Tier 2 -- Moderate signal** (5+ users OR p < 0.10):\n")
if moderate:
    md_parts.extend([f"- {s}" for s in moderate])
else:
    md_parts.append("- *None*")
md_parts.append("\n**Tier 3 -- Preliminary signal** (fewer data points):\n")
if preliminary:
    md_parts.extend([f"- {s}" for s in preliminary])
else:
    md_parts.append("- *None*")

display(Markdown("\n".join(md_parts)))

In [ ]:
# ---------- Visual summary: diverging bar of POTS treatments ----------
vis_df = drug_sum_df.head(15).sort_values("pct_pos").copy()
drugs_v = vis_df["drug"].str.title().values
pct_pos_v = vis_df["pct_pos"].values
# Compute negative pct from user-drug data
neg_pct_v = []
for _, row in vis_df.iterrows():
    grp = pots_user_drug[pots_user_drug["drug"] == row["drug"]]
    n_neg = (grp["outcome"] == "negative").sum()
    neg_pct_v.append(n_neg / len(grp) * 100)
neg_pct_v = np.array(neg_pct_v)
mix_pct_v = 100 - pct_pos_v - neg_pct_v
y_v = np.arange(len(drugs_v))

fig, ax = plt.subplots(figsize=(12, max(5, len(drugs_v) * 0.4)))
ax.barh(y_v, -mix_pct_v, height=0.6, color="#d5d8dc",
        label="Mixed/Neutral", edgecolor="white", linewidth=0.5)
ax.barh(y_v, -neg_pct_v, height=0.6, left=-mix_pct_v, color="#e74c3c",
        label="Negative", edgecolor="white", linewidth=0.5)
ax.barh(y_v, pct_pos_v, height=0.6, color="#2ecc71",
        label="Positive", edgecolor="white", linewidth=0.5)
ax.axvline(x=0, color="black", linewidth=0.8)
ax.set_yticks(y_v)
ax.set_yticklabels(drugs_v, fontsize=10)
for i, row in enumerate(vis_df.itertuples()):
    ax.text(max(pct_pos_v) + 3, i, f"n={row.n_users}", va="center", fontsize=9, color="gray")

ax.set_xlabel("\u2190 % Negative          % Positive \u2192", fontsize=11)
ax.set_title("Treatment Outcomes Among POTS Patients (Top 15, user-level)", fontsize=13)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False, fontsize=10)
ax.spines["left"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

**What this chart shows:** The top 15 treatments among POTS patients, ranked by positive sentiment rate. Green = positive, gray = mixed/neutral, red = negative. This gives a visual overview of which treatments the POTS community finds most helpful.

## 7. Summary + Disclaimer

### Key Findings

**Q1: Ivabradine vs other rate-control drugs.** All rate-control drugs (ivabradine, beta blockers, propranolol) show similar community sentiment around 80% positive. No single rate-control drug clearly outperforms the others.

**Q2: Multi-drug vs single-drug for POTS.** POTS patients trying multiple treatments report higher average sentiment than those on monotherapy. Multi-drug patients average roughly +0.30 compared to roughly -0.39 for single-drug patients. This mirrors the RECOVER-AUTONOMIC finding that multi-prong beats monotherapy.

**Q3: What POTS patients are stacking.** The most common treatment combinations pair rate-control drugs with antihistamines, electrolytes, magnesium, CoQ10, and NAC. The pattern is not about finding the single best drug but about building a complementary stack.

**Bottom line:** The community data independently confirms what RECOVER-AUTONOMIC found in a clinical setting: no single drug fixes POTS. The signal is in the stack.

---

### Reporting-Bias Disclaimer

**Important limitations:**

- This analysis is based entirely on self-selected Reddit posts. Users who never posted about a treatment are not represented.
- Sentiment is extracted by an LLM pipeline and reflects reporting tone, not clinical efficacy.
- Multi-drug users may have more severe illness (prompting them to try more treatments), introducing confounding.
- Sample sizes for individual POTS treatments are modest (3-13 users per drug). Statistical power is limited.
- The 1-month data window may capture seasonal or trending effects.
- Correlation between treatment breadth and sentiment does not prove that adding more drugs causes better outcomes.

**This is not medical advice.** Treatment decisions should be made with a qualified healthcare provider.

In [ ]:
conn.close()
display(HTML("<b>Database connection closed.</b> Analysis complete."))